# Building a Research Agent on Oracle AI Database with Deep Agents, LangChain & Claude

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/langchain_ecosystem/research_agent_with_deepagents_oracle.ipynb)

> **Who this is for:** AI developers who already know how to call an LLM and now want to build a *durable, memory-backed research agent* whose knowledge, working files, and conversation state all live inside **Oracle AI Database** — with **Claude** (Anthropic) doing the reasoning.

## The use case: an incident-research agent

Imagine an on-call engineer types:

> *"Research ORA-50142 and save me a concise, cited brief."*

We want an agent that will, on its own:

1. **Search** an internal knowledge base of incident runbooks and policies that is stored in Oracle — using *hybrid* retrieval (semantic vectors **plus** keyword/full-text, so an exact code like `ORA-50142` is never missed).
2. **Read** the most relevant documents in full.
3. **Plan** its work as a checklist, then **write** a citation-backed brief to a working file.
4. **Remember** everything — the conversation, the file, and reusable notes — so a later thread can pick up where this one left off.

Every one of those verbs is backed by Oracle: the corpus, the vector index, the agent's checkpoints, its long-term notes, and its virtual filesystem. Claude is the brain; Oracle is the long-term memory and the retrieval engine. That is the whole idea of this notebook.

## The agent stack: the **PALO** stack

Throughout this notebook we build on what we'll call the **PALO stack** — the *Palo Alto Agentic Stack* — four layers that each own one concern:

| Letter | Layer | In this notebook | Why it's here |
|:------:|-------|------------------|---------------|
| **P** | **Python** | the notebook runtime & glue | The lingua franca of AI engineering; every layer below has first-class Python bindings. |
| **A** | **Anthropic** | `ChatAnthropic` → **Claude** | The reasoning engine that plans, decides which tools to call, and writes the final brief. |
| **L** | **LangChain** | `langchain-oci`, `deepagents`, `langchain-oracledb`, `langgraph-oracledb` | The orchestration fabric — it turns a chat model into a *tool-using, planning, persistent* agent. |
| **O** | **Oracle** | Oracle AI Database 23ai/26ai | One converged database for vectors, full-text, JSON, agent checkpoints, long-term memory, and files. |

The point of the PALO framing is **separation of concerns**: you can swap the model (A), keep the orchestration (L) and the system of record (O) untouched, and vice-versa. In this notebook we deliberately exercise that seam — the agent factory is provider-agnostic, and we simply hand it a Claude model.

## How the pieces flow

The diagram below traces a single request through the PALO stack. Notice that **Oracle appears four separate times** — as the search corpus, the checkpoint store, the cross-thread memory, and the file backend. That convergence is the reason to build this on Oracle AI Database instead of stitching four services together.

```mermaid
flowchart TD
    U["🧑‍💻 AI developer<br/>'Research ORA-50142 and save a brief'"] --> AGENT

    subgraph L["LangChain orchestration (create_deepagents_agent)"]
        AGENT["Deep Agent loop"]
        PLAN["write_todos<br/>(plan the work)"]
        AGENT --> PLAN
    end

    AGENT -->|"reasons & picks tools"| CLAUDE["🧠 Anthropic Claude<br/>(ChatAnthropic)"]
    CLAUDE -->|"tool calls"| AGENT

    AGENT -->|"search / get_document<br/>(hybrid vector + text)"| CORPUS[("Oracle: research corpus<br/>ADB + Oracle Text index")]
    AGENT -->|"write_file / read_file"| FILES[("Oracle: virtual filesystem<br/>StoreBackend → OracleStore")]
    AGENT -->|"per-thread state"| CKPT[("Oracle: checkpoints<br/>OracleSaver")]
    AGENT -->|"reusable notes"| NOTES[("Oracle: long-term memory<br/>OracleStore")]

    CORPUS --> AGENT
    AGENT --> ANSWER["📄 Cited research brief"]
```

If your renderer doesn't display Mermaid, read the block top-to-bottom: the developer's question enters the LangChain agent loop; Claude decides which tools to call; the agent hits Oracle for search, files, checkpoints, and notes; and a cited brief comes back out.

## What makes it a *Deep* Agent (and not just a chatbot)

A plain **ReAct** agent is a single flat loop: call a tool, read the result, call the next tool, all in one context window. That's fine for short tasks but loses the thread on long, multi-step work.

A **Deep Agent** ([`deepagents`](https://github.com/langchain-ai/deepagents) — [docs](https://docs.langchain.com/oss/python/deepagents/overview)) wraps that same loop in four opinionated capabilities:

- **Planning** via a `write_todos` tool — the agent keeps an explicit, status-tracked checklist so it stays on task.
- **Sub-agents** via a `task` tool — heavy sub-steps run in their own isolated context and return a single result.
- **A virtual filesystem** (`write_file`, `read_file`, `ls`, `edit_file`, `glob`, `grep`) — working memory that lives *outside* the prompt, so intermediate results don't blow the context window.
- **A detailed system prompt** — the operating manual that ties the above together.

We do **not** build the agent with `deepagents.create_deep_agent` directly. Instead we use Oracle's factory, [`langchain_oci.create_deepagents_agent`](https://github.com/oracle/langchain-oracle/tree/main/libs/oci/langchain_oci/agents/deepagents) (from the [`oracle/langchain-oracle`](https://github.com/oracle/langchain-oracle) repo). That factory is a superset: it takes the deepagents machinery **and automatically generates Oracle-backed retrieval tools** (`stats`, hybrid `search`, `get_document`) from whatever datastores you register — so you never hand-write a retrieval tool. That is the "right way" to use deep agents against Oracle, and it's the only agent constructor we call in this notebook.

## The packages we install, and the job each one does

Before any code, here is the full cast. Skim it now; every row shows up later.

| Package | What it is | Role in this notebook |
|---------|-----------|-----------------------|
| [`langchain-oci`](https://pypi.org/project/langchain-oci/) | Oracle's LangChain integration for OCI | Provides `create_deepagents_agent` (the agent factory) and `langchain_oci.datastores.ADB` (the Oracle-backed vector datastore). **The star of the show.** |
| [`deepagents`](https://github.com/langchain-ai/deepagents) | LangChain's deep-agent harness | Supplies the planning/sub-agent/filesystem machinery. We only touch its `StoreBackend` directly (to point the agent's filesystem at Oracle). |
| [`langchain-anthropic`](https://docs.langchain.com/oss/python/integrations/chat/anthropic) | Anthropic chat models for LangChain | `ChatAnthropic` — wraps **Claude**, the reasoning engine that drives the agent. |
| [`langchain-huggingface`](https://docs.langchain.com/oss/python/integrations/providers/huggingface) | HuggingFace embeddings/LLMs for LangChain | `HuggingFaceEmbeddings` — our default, run-anywhere embedding model (`all-MiniLM-L6-v2`, 384-dim). |
| `sentence-transformers` | Local embedding-model runtime | The engine `HuggingFaceEmbeddings` uses to turn text into vectors on your machine. |
| [`langchain-oracledb`](https://pypi.org/project/langchain-oracledb/) | Oracle's LangChain vectorstore + text search | Powers the `ADB` datastore: `OracleVS` vectors, `create_text_index` for Oracle Text, and `OracleEmbeddings` for **in-database** embeddings. |
| `langgraph-oracledb` | Oracle-backed LangGraph persistence | `OracleSaver` (thread checkpoints) and `OracleStore` (long-term semantic memory). |
| `oracledb` | The Python driver for Oracle | Opens the connection pool every layer above shares. |

Everything is a released PyPI package — no local, editable, or VCS installs.

---
# Part 1 — Install the packages

One line, latest releases. `langchain-oci[deepagents]` pulls the agent factory **and** its compatible LangChain / deepagents / Oracle dependencies; we add the model and embedding providers on top. Re-run any time; if pip replaces a package that's already imported, restart the kernel before continuing.

In [ ]:
%pip install -qU \
    "langchain-oci[deepagents]" \
    langchain-anthropic \
    langchain-huggingface \
    sentence-transformers \
    langchain-oracledb \
    langgraph-oracledb \
    oracledb
print("✅ Latest releases installed. Restart the kernel if pip replaced an already-imported package.")

---
# Part 2 — Start Oracle AI Database

The whole PALO **O** layer is one container. We use Gerald Venzl's [`gvenzl/oracle-free`](https://github.com/gvenzl/oci-oracle-free) image, which ships Oracle AI Database Free with the vector engine built in.

> ⚠️ **Do not use a `-slim` tag.** This notebook creates an Oracle **Text** `SEARCH INDEX` for hybrid retrieval, and the slim image omits the Oracle Text component.

Run this once in a terminal (skip if the container already exists — just `docker start oracle-ai-notebooks`):

```bash
docker run --detach --name oracle-ai-notebooks \
  --publish 1521:1521 \
  --env ORACLE_RANDOM_PASSWORD=true \
  --env APP_USER=NOTEBOOK_USER \
  --env APP_USER_PASSWORD=NotebookPwd_2026 \
  --health-cmd=healthcheck.sh --health-interval=10s \
  --health-timeout=5s --health-retries=60 \
  gvenzl/oracle-free:23.26.2

# wait until healthy
until [ "$(docker inspect --format='{{.State.Health.Status}}' oracle-ai-notebooks)" = healthy ]; do sleep 5; done
```

`APP_USER` creates the `NOTEBOOK_USER` schema inside the `FREEPDB1` pluggable database. To point at a different/remote database instead, override `ORACLE_USER`, `ORACLE_PASSWORD`, and `ORACLE_DSN` in Part 3 — and never commit real credentials.

---
# Part 3 — Connect to Oracle

Continuing from the container we just started, we open a single **connection pool** with the [`python-oracledb`](https://pypi.org/project/oracledb/) driver. A pool (rather than a lone connection) matters here because several PALO **O** components — `OracleSaver`, `OracleStore`, and the agent — will borrow connections concurrently.

The `oracledb.create_pool` arguments we set:

- `user`, `password`, `dsn` — the schema and where to reach it. `dsn="localhost:1521/FREEPDB1"` is `host:port/service_name`.
- `min` / `max` — the pool keeps at least `min` connections open and grows to `max` under load.
- `increment` — how many connections to add at a time when the pool needs to grow.

In [2]:
import os

# Credentials come from the environment so nothing secret is committed.
ORACLE_USER = os.getenv("ORACLE_USER", "NOTEBOOK_USER")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD", "NotebookPwd_2026")
ORACLE_DSN = os.getenv("ORACLE_DSN", "localhost:1521/FREEPDB1")

print("Connecting as", ORACLE_USER, "to", ORACLE_DSN)

Connecting as NOTEBOOK_USER to localhost:1521/FREEPDB1


In [3]:
import oracledb

pool = oracledb.create_pool(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    min=1,
    max=6,
    increment=1,
)

# Smoke-test the pool: borrow one connection and confirm who/where we are.
with pool.acquire() as connection, connection.cursor() as cursor:
    current_user, service_name = cursor.execute(
        "SELECT SYS_CONTEXT('USERENV','CURRENT_USER'), "
        "SYS_CONTEXT('USERENV','SERVICE_NAME') FROM dual"
    ).fetchone()

print(f"✅ Connected as {current_user} to {service_name}.")

✅ Connected as NOTEBOOK_USER to freepdb1.


---
# Part 4 — Choose an embedding model

Search only works if text becomes vectors. An **embedding model** does that conversion, and the same model must be used to index the corpus (Part 5) and to embed the agent's long-term notes (Part 6). We support two real backends — pick one with the `EMBEDDING_BACKEND` switch below.

### Option A (recommended for production): **in-database embeddings**

Oracle can generate embeddings *inside the database* from a pretrained ONNX model, so your text never leaves Oracle to be vectorized. This is the most secure and operationally simple option — one system, no external embedding service. The LangChain wrapper is [`OracleEmbeddings`](https://docs.langchain.com/oss/python/integrations/embeddings/oracleai), which calls Oracle's [`DBMS_VECTOR_CHAIN.UTL_TO_EMBEDDINGS`](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/utl_to_embedding-and-utl_to_embeddings-dbms_vector_chain.html) under the hood.

**Setup — where the embeddings come from (all the links you need):**

1. **Get a pretrained ONNX model.** Oracle publishes a ready-to-use, augmented `all-MiniLM-L12-v2` model (384-dim, ~117 MB): **[download `all_MiniLM_L12_v2_augmented.zip`](https://adwc4pm.objectstorage.us-ashburn-1.oci.customer-oci.com/p/TtH6hL2y25EypZ0-rrczRZ1aXp7v1ONbRBfCiT-BDBN8WLKQ3lgyW6RxCfIFLdA6/n/adwc4pm/b/OML-ai-models/o/all_MiniLM_L12_v2_augmented.zip)** (announced on the [Oracle ML blog](https://blogs.oracle.com/machinelearning/use-our-prebuilt-onnx-model-now-available-for-embedding-generation-in-oracle-database-23ai)). Background: [Import Pretrained Models in ONNX Format](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/import-pretrained-models-onnx-format-vector-generation-database.html).
2. **Grant the loading privileges** and create a directory object (full worked example: [Import ONNX Models — End-to-End](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/import-onnx-models-oracle-ai-database-end-end-example.html)):
   ```sql
   GRANT DB_DEVELOPER_ROLE TO NOTEBOOK_USER;
   GRANT CREATE MINING MODEL TO NOTEBOOK_USER;
   CREATE OR REPLACE DIRECTORY DM_DUMP AS '/home/oracle/onnx';
   GRANT READ, WRITE ON DIRECTORY DM_DUMP TO NOTEBOOK_USER;
   ```
3. **Load the model** — from a directory with [`DBMS_VECTOR.LOAD_ONNX_MODEL`](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/load_onnx_model-procedure.html), or straight from object storage with [`DBMS_VECTOR.LOAD_ONNX_MODEL_CLOUD`](https://docs.oracle.com/en/database/oracle/oracle-database/23/vecse/load_onnx_model_cloud.html). The Python helper `OracleEmbeddings.load_onnx_model(conn, "DM_DUMP", "all_MiniLM_L12_v2.onnx", "doc_model")` wraps the file-based call.

The reference chapter for all of this is the [Oracle AI Vector Search User's Guide](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/overview-ai-vector-search.html).

### Option B (default here): **HuggingFace embeddings, run locally**

So this notebook runs on any machine with **zero extra database privileges**, the default backend loads a [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) model locally through [`langchain-huggingface`](https://docs.langchain.com/oss/python/integrations/providers/huggingface). It produces the same 384-dim vectors, entirely on your CPU. (Use the modern partner-package import `from langchain_huggingface import HuggingFaceEmbeddings` — the old `langchain_community` class is deprecated.)

### Build the embedding model

Flip `EMBEDDING_BACKEND` to `"oracle"` after completing the Option-A setup above; leave it as `"huggingface"` to run immediately. Either way we end up with a standard LangChain `Embeddings` object bound to the name `embeddings`.

`HuggingFaceEmbeddings` constructor arguments we pass:
- `model_name` — any [Sentence-Transformers model id](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) on the HuggingFace Hub.
- `model_kwargs={"device": "cpu"}` — where the model runs (`"cuda"` if you have a GPU).
- `encode_kwargs={"normalize_embeddings": True}` — L2-normalize vectors so cosine similarity behaves well.

In [4]:
EMBEDDING_BACKEND = os.getenv("EMBEDDING_BACKEND", "huggingface")  # "huggingface" | "oracle"

if EMBEDDING_BACKEND == "oracle":
    # In-database embeddings. Requires a loaded ONNX model (see Option A above).
    # `params.model` is the model name you gave DBMS_VECTOR.LOAD_ONNX_MODEL.
    from langchain_oracledb.embeddings.oracleai import OracleEmbeddings

    embeddings = OracleEmbeddings(
        conn=pool.acquire(),
        params={"provider": "database", "model": os.getenv("ORACLE_ONNX_MODEL", "doc_model")},
    )
else:
    # Default: local HuggingFace model, no DB privileges needed.
    from langchain_huggingface import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

print(f"Embedding backend: {EMBEDDING_BACKEND}")

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_eco/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8305.23it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding backend: huggingface


Before we build any tables we need two facts derived from the model: its **vector dimensionality** (Oracle's vector columns and indexes are dimension-typed) and a set of **run-scoped names**. We embed one probe string to read the dimension straight from the model, and mint a random `RUN_ID` so re-running the notebook never collides with a previous run's tables, namespaces, or checkpoint threads.

In [5]:
import uuid

# Read the true output dimension straight from the model.
EMBED_DIMS = len(embeddings.embed_query("dimension probe"))

# Unique, collision-free names for this run.
RUN_ID = uuid.uuid4().hex[:10].upper()
ADB_TABLE = f"ADB_RESEARCH_{RUN_ID}"
TEXT_INDEX_NAME = f"ADB_TXT_{RUN_ID}"
STORE_SUFFIX = f"research_{RUN_ID.lower()}"
ROOT_NAMESPACE = ("deepagents_oracle", RUN_ID.lower())
NOTES_NAMESPACE = ROOT_NAMESPACE + ("notes",)
FILES_NAMESPACE = ROOT_NAMESPACE + ("files",)
RESEARCH_THREAD = f"research-{RUN_ID.lower()}"

print(f"Embedding dimensions: {EMBED_DIMS}")
print(f"Run ID: {RUN_ID}  ->  table {ADB_TABLE}, thread {RESEARCH_THREAD}")

Embedding dimensions: 384
Run ID: 923F9F3C98  ->  table ADB_RESEARCH_923F9F3C98, thread research-923f9f3c98


---
# Part 5 — Seed the Oracle research corpus

Now we give the agent something to research. The corpus is a small set of incident runbooks and one policy — the internal knowledge the agent will search instead of guessing.

We store it through [`langchain_oci.datastores.ADB`](https://github.com/oracle/langchain-oracle), the Oracle-backed vector datastore. `ADB` wraps [`OracleVS`](https://docs.langchain.com/oss/python/integrations/vectorstores/oracle) (vector storage + similarity search) and adds the hooks the agent factory needs. Its constructor arguments:

- `dsn`, `user`, `password` — the same Oracle connection details from Part 3 (`ADB` opens its own connection for bulk loading).
- `table_name` — the table that will hold the documents and their vectors (our run-scoped `ADB_TABLE`).
- `datastore_description` — a natural-language description of what lives here. The agent factory embeds this so it can **auto-route** a query to the right datastore when you register several.
- `chunk_on_write=False` — keep each short runbook as one row instead of splitting it; our documents are already bite-sized.

In [6]:
RESEARCH_DOCUMENTS = [
    {
        "id": "runbook-ora-50142",
        "title": "Connection pool exhaustion runbook",
        "content": (
            "ORA-50142 indicates an exhausted application connection pool. "
            "Check leaked sessions, cap checkout time, recycle the pool, and "
            "confirm recovery before closing the incident."
        ),
        "source": "internal-runbook",
    },
    {
        "id": "runbook-vector-latency",
        "title": "Vector search latency runbook",
        "content": (
            "Inspect execution plans, embedding dimensions, HNSW neighbor "
            "settings, and application top-k values for vector latency."
        ),
        "source": "internal-runbook",
    },
    {
        "id": "policy-severity",
        "title": "Incident severity policy",
        "content": (
            "A production outage blocking customers is severity P1. Assign an "
            "incident commander and publish updates every 30 minutes."
        ),
        "source": "internal-policy",
    },
]
print(f"{len(RESEARCH_DOCUMENTS)} documents ready to load.")

3 documents ready to load.


We instantiate the datastore, then `connect(embeddings)` to bind the embedding model we chose in Part 4 (this is where `ADB` learns how to vectorize text). Finally `bulk_insert` writes the rows: it takes the document dicts **and** their precomputed vectors — we embed the `content` field of each document with the very same `embeddings` object, guaranteeing the corpus and future queries share one vector space.

In [7]:
from langchain_oci.datastores import ADB

research_adb = ADB(
    dsn=ORACLE_DSN,
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    table_name=ADB_TABLE,
    datastore_description=(
        "Oracle incident runbooks, exact ORA codes, vector operations, "
        "and production severity policy"
    ),
    chunk_on_write=False,
)
research_adb.connect(embeddings)

inserted = research_adb.bulk_insert(
    RESEARCH_DOCUMENTS,
    embeddings=embeddings.embed_documents(
        [doc["content"] for doc in RESEARCH_DOCUMENTS]
    ),
)
print(f"✅ Inserted {inserted} rows into {ADB_TABLE}.")

✅ Inserted 3 rows into ADB_RESEARCH_923F9F3C98.


## 5.1 Add an Oracle Text index for **hybrid** search

Vectors are great at *meaning* ("exhausted pool" ≈ "pool ran out") but can miss *exact tokens* like the error code `ORA-50142`. So we add a keyword index with [`create_text_index`](https://docs.langchain.com/oss/python/integrations/vectorstores/oracle), which builds an Oracle **Text** `SEARCH INDEX` over the same table. The generated `search` tool then fuses both signals with reciprocal-rank fusion — semantic recall *and* exact-match precision.

`create_text_index` arguments:
- `client` — the live Oracle connection the index DDL runs on (`research_adb.vectorstore.client`).
- `idx_name` — the index name (our run-scoped `TEXT_INDEX_NAME`).
- `vector_store` — the `OracleVS` instance whose table gets indexed.

After building it we `close()` the datastore's loader connection — the agent factory re-opens its own when it runs.

In [8]:
from langchain_oracledb.retrievers.text_search import create_text_index

create_text_index(
    client=research_adb.vectorstore.client,
    idx_name=TEXT_INDEX_NAME,
    vector_store=research_adb.vectorstore,
)
research_adb.close()
print(f"✅ Hybrid search ready: vector table + Oracle Text index {TEXT_INDEX_NAME}.")

✅ Hybrid search ready: vector table + Oracle Text index ADB_TXT_923F9F3C98.


---
# Part 6 — Give the agent memory in Oracle

The corpus is what the agent *reads*. This part is what the agent *remembers*. Three [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence) components — all Oracle-backed — cover three different memory horizons:

| Component | Memory horizon | Backed by |
|-----------|----------------|-----------|
| `OracleSaver` | **Short-term** — the full message history of one thread, so a run can resume exactly where it stopped | checkpoint tables |
| `OracleStore` | **Long-term** — semantic notes that any thread can recall by meaning | a vector-indexed store |
| `StoreBackend` | The agent's **virtual filesystem** — `write_file`/`read_file` mapped onto `OracleStore` | a namespace in `OracleStore` |

They all share the connection `pool` from Part 3.

### OracleStore — long-term semantic memory

`OracleStore` is a vector-searchable key/value store. Its `index` argument tells it how to embed and index note text:
- `dims` — vector size, our `EMBED_DIMS`.
- `embed` — the embedding model (the same `embeddings` again).
- `fields` — which fields of a stored value to embed (`["text"]`).
- `index_type` — the ANN index: an `hnsw` graph with `neighbors`/`efconstruction` build parameters and `COSINE` distance.

`table_suffix` keeps this run's store tables separate. `setup()` creates the tables if they don't exist.

In [9]:
from langgraph_oracledb.store.oracle import OracleStore

STORE_INDEX = {
    "dims": EMBED_DIMS,
    "embed": embeddings,
    "fields": ["text"],
    "index_type": {
        "type": "hnsw",
        "neighbors": 16,
        "efconstruction": 64,
        "distance_metric": "COSINE",
    },
}
store = OracleStore(pool, index=STORE_INDEX, table_suffix=STORE_SUFFIX)
store.setup()
print("✅ OracleStore ready (long-term semantic memory).")

✅ OracleStore ready (long-term semantic memory).


### OracleSaver — per-thread checkpoints

`OracleSaver` snapshots the agent's graph state after each step, keyed by `thread_id`. Resuming a thread replays its last checkpoint, so the agent continues with full history.

We pass `json_size_threshold_mb=0.0` to route **all** channel values through the saver's serializer rather than the database's native JSON binding — that keeps complex LangGraph state compatible with `python-oracledb` 4.x's strict JSON handling. `setup()` creates the checkpoint tables.

In [10]:
from langgraph_oracledb.checkpoint.oracle import OracleSaver

checkpointer = OracleSaver(pool, json_size_threshold_mb=0.0)
checkpointer.setup()
print("✅ OracleSaver ready (per-thread checkpoints).")

✅ OracleSaver ready (per-thread checkpoints).


### StoreBackend — the agent's filesystem, on Oracle

`StoreBackend` (from `deepagents.backends`) is the one deepagents primitive we wire up by hand. It implements the deep agent's virtual filesystem on top of `OracleStore`:
- `store` — the `OracleStore` to persist into.
- `namespace` — a callable returning the namespace tuple for files; we pin it to `FILES_NAMESPACE` so files land in their own partition, separate from notes.
- `file_format="v2"` — the current on-disk file schema.

We also seed one **cross-thread note** directly into `OracleStore` (`store.put(namespace, key, value)`) — later we'll prove a brand-new thread can recall it by meaning alone.

In [11]:
from deepagents.backends import StoreBackend

backend = StoreBackend(
    store=store,
    namespace=lambda _runtime: FILES_NAMESPACE,
    file_format="v2",
)

store.put(
    NOTES_NAMESPACE,
    "ora-50142-note",
    {
        "text": (
            "Recover ORA-50142 by checking leaked sessions, recycling the "
            "connection pool, and verifying health."
        ),
        "title": "Shared pool-exhaustion note",
        "source_thread": RESEARCH_THREAD,
    },
)
print("✅ StoreBackend ready and one cross-thread note seeded.")

✅ StoreBackend ready and one cross-thread note seeded.


---
# Part 7 — Assemble the Deep Agent

Every PALO layer is now in place: **O**racle holds the corpus and the memory, and we're about to add **A**nthropic (the model) through **L**angChain (the factory). This is the payoff cell.

### The model — Claude via `ChatAnthropic`

We build the model ourselves and hand it to the factory. Arguments:
- `model` — a current Claude id; we default to `claude-sonnet-5` (override with `ANTHROPIC_MODEL`). It's capable enough to plan multi-tool research yet cheaper than Opus for a teaching run.
- `max_tokens` — cap on the response length.

Note we **don't** pass `temperature`: the newest Claude models manage sampling internally and reject the parameter. The key is read from the `ANTHROPIC_API_KEY` environment variable — never hard-code it.

In [12]:
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your ANTHROPIC_API_KEY: ")

from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5"),
    max_tokens=2048,
)
print(f"✅ Claude model ready: {model.model}")

✅ Claude model ready: claude-sonnet-5


### The factory — `create_deepagents_agent`

Recall from the *"What makes it a Deep Agent"* section: this Oracle factory turns our datastores into retrieval tools automatically and wraps everything in the deepagents planning/filesystem loop. The arguments we pass connect all four PALO layers in one call:

- `model=model` — our Claude model. When you pass a prebuilt model, the factory uses it **as-is** and ignores its own `model_id`/OCI-auth options (Claude owns its own connection).
- `datastores={"oracle_research": research_adb}` — the datastores to expose. The factory generates `stats`, hybrid `search`, and `get_document` tools for each.
- `default_datastore="oracle_research"` — which datastore to use when routing is ambiguous (we only have one).
- `embedding_model=embeddings` — the embedding model for datastore search. **Important:** the factory's default is an OCI Cohere model; passing `embeddings` here is how we make it use our HuggingFace/Oracle backend instead.
- `top_k=3` — how many documents `search` returns.
- `system_prompt=...` — the agent's operating instructions (search, cite Doc IDs, keep notes under `/research`, never invent citations).
- `checkpointer`, `store`, `backend` — the three Oracle-backed memory components from Part 6.
- `name=...` — a label for the compiled agent graph.

In [13]:
from langchain_oci import create_deepagents_agent

SYSTEM_PROMPT = (
    "You are an incident-research agent. Search the Oracle research datastore, "
    "cite document IDs for every claim, and keep working notes under /research. "
    "Never invent citations."
)

agent = create_deepagents_agent(
    model=model,
    datastores={"oracle_research": research_adb},
    default_datastore="oracle_research",
    embedding_model=embeddings,
    top_k=3,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
    store=store,
    backend=backend,
    name="oracle-research-agent",
)
print("✅ Deep Agent compiled. Tools are auto-generated from the Oracle datastore.")

✅ Deep Agent compiled. Tools are auto-generated from the Oracle datastore.


---
# Part 8 — Run the research agent (live)

Time to ask the question from the very top of the notebook. We `invoke` the agent with the developer's request and a `config` that pins the `thread_id` — that's the key `OracleSaver` uses to checkpoint this conversation.

This is a **real** Claude run. The model will decide, on its own, to call the auto-generated `search` tool, read a document with `get_document`, and persist a brief with the built-in `write_file` tool — no scripted responses anywhere.

In [14]:
result = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": (
            "Research ORA-50142 and save a concise, cited brief to "
            "/research/ora-50142-brief.md."
        ),
    }]},
    config={"configurable": {"thread_id": RESEARCH_THREAD}},
)

print(result["messages"][-1].content)

Brief saved to `/research/ora-50142-brief.md`. Summary:

**ORA-50142** = application connection pool exhaustion [Doc ID: runbook-ora-50142].

- **Cause pattern:** typically leaked sessions holding connections open.
- **Fix steps:** check for leaked sessions → cap checkout time → recycle the pool → confirm recovery before closing the incident [Doc ID: runbook-ora-50142].
- **Severity:** if it's causing a customer-facing production outage, classify as **P1** — assign an incident commander, publish updates every 30 min [Doc ID: policy-severity].

Only 2 of the 3 documents in the datastore were relevant; a vector-latency runbook [Doc ID: runbook-vector-latency] was reviewed but excluded as inapplicable.


Let's confirm which tools Claude actually chose. Each `ToolMessage` in the returned message list is one tool result; printing their names in order shows the agent's real trajectory (typically `search` → `get_document` → `write_file`).

In [15]:
from langchain_core.messages import ToolMessage

tool_sequence = [m.name for m in result["messages"] if isinstance(m, ToolMessage)]
print("Tools Claude called, in order:", tool_sequence)

Tools Claude called, in order: ['search', 'stats', 'get_document', 'get_document', 'write_file']


---
# Part 9 — Inspect what Oracle persisted

The run is over, but nothing was lost — it's all in Oracle. We now open each memory horizon from Part 6 and read it back, proving the durability the PALO **O** layer buys us.

### 9.1 The file the agent wrote

We build a *fresh* `StoreBackend` over the same namespace — a stand-in for a different process reconnecting later — and read the brief the agent saved. It comes back from `OracleStore`, not from any in-memory state.

In [16]:
fresh_backend = StoreBackend(
    store=store,
    namespace=lambda _runtime: FILES_NAMESPACE,
    file_format="v2",
)
saved = fresh_backend.read("/research/ora-50142-brief.md")
print("Read error:", saved.error)
print("-" * 60)
print(saved.file_data["content"])

Read error: None
------------------------------------------------------------
# Incident Brief: ORA-50142

## Summary
ORA-50142 indicates that an application-level connection pool has been exhausted — the application has run out of available database connections to hand out to requesters [Doc ID: runbook-ora-50142].

## Root Cause Pattern
Connection pool exhaustion is typically driven by leaked sessions (connections checked out and never returned) rather than raw database capacity limits [Doc ID: runbook-ora-50142].

## Remediation Steps
Per the connection pool exhaustion runbook [Doc ID: runbook-ora-50142]:
1. **Check for leaked sessions** — identify connections held open beyond expected use.
2. **Cap checkout time** — enforce a maximum duration a connection can be checked out to prevent indefinite holds.
3. **Recycle the pool** — restart/refresh the pool to clear stuck or leaked connections.
4. **Confirm recovery before closing the incident** — verify the pool returns to healthy avai

### 9.2 Cross-thread semantic memory

Here's the memory payoff. We query `OracleStore` from a *different* angle than the note was written — no keyword overlap — and it still surfaces our seeded note by **meaning**. `store.search(namespace, query, limit)` returns ranked hits with a similarity `score`. Any future thread, agent, or process sharing this store can recall it.

In [17]:
hits = store.search(
    NOTES_NAMESPACE,
    query="how do I recover an exhausted Oracle connection pool?",
    limit=3,
)
for hit in hits:
    print(f"[{hit.score:.3f}] {hit.key} — {hit.value.get('title')}")

[0.585] ora-50142-note — Shared pool-exhaustion note


### 9.3 The conversation checkpoint

Finally, `OracleSaver` kept the whole thread. `agent.get_state(config)` replays the last checkpoint for our `thread_id`; the message count shows the full trajectory (human turn + Claude's tool calls + tool results + final answer) is safely on disk. Re-invoking the agent with the same `thread_id` would continue this exact conversation.

In [18]:
state = agent.get_state({"configurable": {"thread_id": RESEARCH_THREAD}})
print(f"Thread {RESEARCH_THREAD!r} has {len(state.values['messages'])} persisted messages.")
print("Latest checkpoint id:", state.config["configurable"]["checkpoint_id"])

Thread 'research-923f9f3c98' has 10 persisted messages.
Latest checkpoint id: 1f17a052-c48e-69fc-800c-ea50da5d3343


---
# Part 10 — Clean up

Because every name is run-scoped, cleanup is surgical. We drop this run's ADB table (its Oracle Text index goes with it), delete the checkpoint thread, tear down the run's `OracleStore` tables, and close the pool. Set `KEEP_ORACLE_NOTEBOOK_TABLES=1` to keep the tables for inspection while still removing the checkpoint rows.

In [19]:
KEEP_ORACLE_NOTEBOOK_TABLES = os.getenv("KEEP_ORACLE_NOTEBOOK_TABLES", "0").lower() in {"1", "true", "yes"}

# 1) Remove this thread's checkpoints.
checkpointer.delete_thread(RESEARCH_THREAD)
print("Deleted checkpoint thread:", RESEARCH_THREAD)

# 2) Drop the ADB table (+ its Text index) and the OracleStore tables.
if KEEP_ORACLE_NOTEBOOK_TABLES:
    print(f"Keeping {ADB_TABLE} and OracleStore suffix {STORE_SUFFIX} for inspection.")
else:
    with pool.acquire() as connection, connection.cursor() as cursor:
        try:
            cursor.execute(f"DROP TABLE {ADB_TABLE} PURGE")
            print(f"Dropped {ADB_TABLE} (and index {TEXT_INDEX_NAME}).")
        except oracledb.Error as exc:
            if getattr(exc.args[0], "code", None) != 942:  # 942 = table doesn't exist
                raise
    store.teardown()
    print(f"Removed OracleStore tables for {STORE_SUFFIX}.")

# 3) Close the shared pool.
pool.close(force=True)
print("✅ Cleanup complete; Oracle connection pool closed.")

Deleted checkpoint thread: research-923f9f3c98


Dropped ADB_RESEARCH_923F9F3C98 (and index ADB_TXT_923F9F3C98).
Removed OracleStore tables for research_923f9f3c98.
✅ Cleanup complete; Oracle connection pool closed.


---
# Recap & where to go next

You built the full **PALO** stack end to end:

- **P**ython glued it together; **A**nthropic's Claude did the reasoning; **L**angChain's `create_deepagents_agent` turned the model into a planning, tool-using agent; and **O**racle AI Database was the single system of record for the corpus, the vectors, the checkpoints, the long-term notes, and the agent's files.
- The agent **auto-generated** its own hybrid-search tools from an Oracle datastore, ran a **real** multi-step Claude trajectory (`search → get_document → write_file`), and everything it touched **survived** the run inside Oracle.

**Ideas to extend it:**
- **Go fully in-database**: complete the Part 4 Option-A setup and flip `EMBEDDING_BACKEND="oracle"` so text is vectorized inside Oracle — see the [Oracle AI Vector Search User's Guide](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/overview-ai-vector-search.html).
- **Register more datastores**: add a second `ADB` (or an `OpenSearch`) with its own `datastore_description` and watch the factory **auto-route** between them.
- **Add sub-agents**: pass `subagents=[...]` to `create_deepagents_agent` to delegate heavy sub-tasks into isolated contexts.
- **Swap the model** (the PALO seam): change only the `A` layer — e.g. a different Claude tier via `ANTHROPIC_MODEL` — and leave the rest untouched.

**Reference docs:** [deepagents](https://docs.langchain.com/oss/python/deepagents/overview) · [langchain-oci (Oracle)](https://github.com/oracle/langchain-oracle) · [ChatAnthropic](https://docs.langchain.com/oss/python/integrations/chat/anthropic) · [OracleVS vectorstore](https://docs.langchain.com/oss/python/integrations/vectorstores/oracle) · [OracleEmbeddings](https://docs.langchain.com/oss/python/integrations/embeddings/oracleai) · [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence).